Sheet 3.1: Transformer (1) - Attention
===
**Author**: Amir Mohammadpour

The transformer, the star of current season of the AI show, hinges upon one simple machanism: Attention! And attention was once a mere afterthought. Researchers working on sequence-to-sequence tasks, especially machine translation, were already using attention layers on top of recurrent networks. A typical system combined an encoder-decoder structure with attention to allow the decoder to access all encoder states. This improved performance, but the recurrent backbone remained the limiting factor.

And recurrence enforeces sequential execution. Back propagation through time, though seemingly inacous in our tiny model from previous session, is a techincal failure at scale. Even with GPUs, you cannot parallelize across time steps. LSTMs and GRUs, although successfull in adressing vanishing gradient problem that RNNs introduced, were still constrained by the factor of time.

So at some point, some people started to ask a simple question: What if the attention is all you need?
The recurrence seemed like an old bureaucrat, stamping forms no one ever reads. What if we remove it and nothing breaks?

You have already been presented with the theory of what attentiom, here we take another approach and try to debunk the mystery of attention mechanism in action.

---
## 1. What attention computes

So how does attention replace recurrence without its drawbacks? The hidden state in a recurrent network served a single purpose: to make each token context-aware. At every time step, it summarized everything seen so far and passed it forward. Attention replaces this mechanism entirely — but the goal remains the same. Each token must be represented not in isolation, but in the context of the sequence it belongs to.

The simplest way to achieve this is also the most naive: at each position, average over all tokens. Every token contributes equally to the current representation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Say we have a weight matrix `W` of shape `(3, 3)` and a value matrix `V` of shape `(3, 2)`. Each row of `V` represents a token embedding. How can we compute the weighted sum of these tokens? Well, how about a loop?

In [ ]:
V = torch.tensor([
    [1., 2.],
    [3., 4.],
    [5., 6.],
])

W = torch.tensor([
    [1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0],
])

out = torch.zeros(W.shape[0], V.shape[1])
for i in range(W.shape[0]):
    for j in range(V.shape[1]):
        out[i, j] = (W[i] * V[:, j]).mean()
print(out)

But we know from linear algebra that we can easily achieve what the loop above does, with a matrix multiplication:

In [ ]:
V = torch.tensor([
    [1., 2.],
    [3., 4.],
    [5., 6.],
])

W = torch.tensor([
    [1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0],
])

W = W / W.sum(dim=-1, keepdim=True)  # normalize rows to sum to 1
out = W @ V
print(out)

What we have done thus far is, for each token, take the entirety of the sequence as the *context*, compute a weighted sum over that context, and replace each token with the result. Make sure you get this right because we're getting serious here.
But this is not what you expect from our future text generating language model to do.
> Think about it, what's wrong with this setting?

Now consider what happens when `W` is a lower triangular matrix with uniform weights. The `torch.tril` function zeros out everything above the diagonal, leaving only past and present positions. If we then normalize each row to sum to 1, the matrix multiplication turns into an averaging operation, for all *preceding* positions.

In [ ]:
tril = torch.tril(torch.ones(3, 3))
print(tril)

In [ ]:
W = tril / tril.sum(dim=-1, keepdim=True)
print(W)

In [ ]:
out = W @ V
print(out)

Notice how each row is the average of tokens up until that position. You should be able to confirm that the first row is just `V[0]`, the second row is the mean of `V[0]` and `V[1]`, and the third row is the mean of `V[0]`, `V[1]`, and `V[2]`.

One clarification before we proceed: throughout this section, `V` has been playing a dual role — its rows are both the raw input token embeddings *and* the vectors being aggregated by `W`. That conflation was harmless here, but it will become important to distinguish the two shortly. From this point on, we will use `X` to denote the matrix of input token embeddings, and reserve `V` for a specific learned projection of `X`.

> There is a second problem with the setup above that we have not yet addressed. The mask fixed *which* positions each token can attend to, but the weights within that allowed region are still uniform — every preceding token contributes equally, regardless of content. Is that a reasonable assumption for a language model? What would a better `W` look like, and where should it come from?

---
## 2. From fixed weights to learned attention

The weight matrix `W` we have been using was constructed by hand: uniform, then masked. Nothing about it depends on the actual content of the sequence. Real attention replaces this with a data-dependent weight matrix — one that is computed from the input itself, so that tokens can attend selectively based on what they contain.

The mechanism is parameterized by three learned linear projections of the input `X`, conventionally denoted Q, K, and V — queries, keys, and values. The names are borrowed from information retrieval: a query is what a position is looking for; a key is what each position advertises about itself; a value is what each position contributes if selected. For a given query, the attention weights are computed by matching it against all keys, normalizing those match scores into a distribution, and taking the weighted sum of the corresponding values.

### 2.1 The projections

Let `X` be a matrix of shape `(T, d_model)`, where `T` is the sequence length and `d_model` is the embedding dimension. We define three weight matrices — `W_Q`, `W_K`, and `W_V` — each of shape `(d_model, d_k)`, and project `X` through each of them. The resulting matrices Q, K, and V each have shape `(T, d_k)`.

In [ ]:
torch.manual_seed(42)

T = 4        # sequence length
d_model = 8  # embedding dimension
d_k = 4      # query/key/value dimension

X = torch.randn(T, d_model)  # input token embeddings, shape (T, d_model)

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

Q = X @ W_Q  # (T, d_k)
K = X @ W_K  # (T, d_k)
V = X @ W_V  # (T, d_k)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

### 2.2 Computing attention scores

To determine how much each position should attend to every other position, we compute pairwise dot products between queries and keys. The dot product of two vectors measures their alignment — a large positive value indicates that a query and key point in similar directions in the embedding space, and therefore that the corresponding positions are, under the learned projections, relevant to each other.

For all pairs simultaneously, this is a single matrix multiplication: `Q @ K.T`, which produces a matrix of shape `(T, T)`. Entry `[i, j]` is the raw attention score of position `i` attending to position `j`.

In [ ]:
scores = Q @ K.T  # (T, T)
print(f"Scores shape: {scores.shape}")
print(scores)

### 2.3 Scaling

Before normalizing these scores into a distribution, we scale them by $\frac{1}{\sqrt{d_k}}$. The reason is purely about variance: the dot product of two random vectors of dimension $d_k$ has variance proportional to $d_k$. Without scaling, large values of $d_k$ push the scores into regions where the softmax saturates — that is, where one entry dominates and the gradient nearly vanishes. Dividing by $\sqrt{d_k}$ returns the scores to unit variance regardless of head dimension.

In [ ]:
import math

scores = scores / math.sqrt(d_k)
print(scores)

### 2.4 Causal masking

We now need to enforce causality: position `i` must not attend to any position `j > i`. In Section 1, we achieved this by zeroing out the upper triangle of `W`. Here, however, the scores will be passed through a softmax — and zeroing them out before softmax does not prevent those positions from receiving some nonzero probability weight. 

> How should we set the masked positions so that they contribute exactly zero probability after softmax? What value, when passed through $e^x$ and then normalized, produces zero?

In [ ]:
tril = torch.tril(torch.ones(T, T))

# Positions where tril is 0 (i.e., future positions) are filled with -inf.
# After softmax, exp(-inf) = 0, so these positions receive zero weight exactly.
scores = scores.masked_fill(tril == 0, float('-inf'))
print(scores)

### 2.5 Softmax: the new W

With the mask in place, we apply softmax row-wise. Each row of the resulting matrix sums to 1, the `-inf` entries become exactly 0, and the remaining entries reflect the relative relevance of each past position to the current query. This is the data-dependent, causal weight matrix that we set out to construct — the successor to the handcrafted `W` from Section 1.

In [ ]:
W = F.softmax(scores, dim=-1)
print(W)

Notice the structure: row `i` has nonzero weights only at positions `0` through `i`, and those weights sum to 1. The pattern is exactly what we constructed manually in Section 1 — but now the weights are not uniform. They are learned from the content of `X` via the Q and K projections.

The output is then computed exactly as before — a matrix multiplication of the weight matrix against the values:

In [ ]:
out = W @ V  # (T, d_k)
print(f"Output shape: {out.shape}")
print(out)

This is scaled dot-product attention in full. To summarize the computation concisely:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

where $M$ is the causal mask matrix — zero on and below the diagonal, $-\infty$ above it. The entire pipeline can be written in a handful of lines:

In [ ]:
def attention(Q, K, V):
    d_k = Q.shape[-1]
    T = Q.shape[-2]
    tril = torch.tril(torch.ones(T, T))
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    scores = scores.masked_fill(tril == 0, float('-inf'))
    W = F.softmax(scores, dim=-1)
    return W @ V

out = attention(Q, K, V)
print(out.shape)

---
## 3. Adding the batch dimension

In practice, inputs arrive not as a single sequence but as a batch of sequences processed in parallel. `X` therefore has shape `(B, T, d_model)`, where `B` is the batch size. The weight matrices `W_Q`, `W_K`, `W_V` are shared across the batch — each sequence is projected independently with the same parameters.

PyTorch's `@` operator handles this transparently: when applied to tensors with more than two dimensions, it treats all leading dimensions as batch dimensions and performs the matrix multiplication over the last two. No other change to the computation is required.

In [ ]:
torch.manual_seed(42)

B = 2        # batch size
T = 4        # sequence length
d_model = 8
d_k = 4

X = torch.randn(B, T, d_model)  # (B, T, d_model)

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

Q = X @ W_Q  # (B, T, d_k)
K = X @ W_K  # (B, T, d_k)
V = X @ W_V  # (B, T, d_k)

out = attention(Q, K, V)  # (B, T, d_k)
print(f"Output shape: {out.shape}")

The output has shape `(B, T, d_k)` — one context-aware representation per token per sequence in the batch. The `attention` function we wrote above already handles this correctly, because `K.transpose(-2, -1)` transposes only the last two dimensions, leaving the batch dimension untouched.

---
## 4. Wrapping it as an `nn.Module`

Following the course convention, we first implement the full mechanism from scratch as an `nn.Module`, making the learned projections explicit. We then compare it to PyTorch's built-in equivalent.

A standard implementation also projects the output back to `d_model` after the weighted sum, via a final linear layer `W_O`. This is included below — it allows the attention head's internal dimension `d_k` to differ from the model dimension, and it is where the head learns how to write its result back into the residual stream.

Note also that `nn.Linear` subsumes the weight matrices `W_Q`, `W_K`, `W_V`, and `W_O` — each `Linear(d_model, d_k, bias=False)` is exactly the matrix multiplication `X @ W.T` we have been writing by hand.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, d_k, T):
        super().__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.W_O = nn.Linear(d_k, d_model, bias=False)
        # register the causal mask as a buffer so it moves with the module to GPU etc.
        self.register_buffer('tril', torch.tril(torch.ones(T, T)))

    def forward(self, X):
        # X: (B, T, d_model)
        Q = self.W_Q(X)  # (B, T, d_k)
        K = self.W_K(X)  # (B, T, d_k)
        V = self.W_V(X)  # (B, T, d_k)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, T, T)
        scores = scores.masked_fill(self.tril == 0, float('-inf'))
        W = F.softmax(scores, dim=-1)                              # (B, T, T)

        out = W @ V        # (B, T, d_k)
        out = self.W_O(out)  # (B, T, d_model)
        return out


torch.manual_seed(42)
X = torch.randn(B, T, d_model)
attn = CausalSelfAttention(d_model=d_model, d_k=d_k, T=T)
out = attn(X)
print(f"Output shape: {out.shape}")  # (B, T, d_model)

### 4.1 From one head to many

The single attention head we have built learns one set of Q, K, V projections. That means it learns one way of measuring relevance between positions — one notion of what to look for and what to retrieve. This is a significant constraint. In a sentence like *"The animal didn't cross the street because it was too tired"*, resolving what *it* refers to requires tracking syntactic subject relationships. But the same sentence may simultaneously require tracking semantic properties of *animal* and *tired* to confirm the resolution. A single head must compromise between these different relational structures in a shared projection space.

Multi-head attention removes this constraint by running $H$ independent attention heads in parallel, each with its own learned projections. Each head attends to the sequence in a different way, producing a representation of shape `(B, T, d_k)`. The outputs of all heads are concatenated along the last dimension to give `(B, T, H * d_k)`, and a final linear projection `W_O` maps this back to `(B, T, d_model)`.

The standard convention sets `d_k = d_model // H`, so the total parameter count stays comparable to a single head of full dimension. The heads do not see each other's projections — they are entirely independent until the concatenation step.

In [ ]:
class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads, T):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        # each head is an independent CausalSelfAttention operating at dimension d_k
        self.heads = nn.ModuleList([
            CausalSelfAttention(d_model=d_model, d_k=self.d_k, T=T)
            for _ in range(num_heads)
        ])
        # W_O projects the concatenated head outputs back to d_model
        # note: CausalSelfAttention already has its own W_O projecting d_k -> d_model,
        # so here we redefine heads without that inner projection for clarity
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        # run each head independently: each produces (B, T, d_k)
        head_outputs = [head(X) for head in self.heads]  # H tensors of (B, T, d_model) -- see note below
        # concatenate along the last dimension: (B, T, H * d_model_per_head)
        out = torch.cat(head_outputs, dim=-1)  # (B, T, d_model)
        out = self.W_O(out)                    # (B, T, d_model)
        return out


torch.manual_seed(42)
num_heads = 2
X = torch.randn(B, T, d_model)
mha_scratch = MultiHeadCausalAttention(d_model=d_model, num_heads=num_heads, T=T)
out = mha_scratch(X)
print(f"Output shape: {out.shape}")  # (B, T, d_model)

A note on the implementation above: we are reusing `CausalSelfAttention` as a building block, which internally already applies its own `W_O` projection from `d_k` back to `d_model`. This makes the head outputs directly concatenable, but it means each head's output has already been projected before the final `W_O` — a slight departure from the canonical formulation where `W_O` is the sole output projection. In practice, the two are equivalent in expressive capacity; the difference is an implementation convenience here. A production implementation would typically omit the inner `W_O` per head and apply only the single final projection.

> Given that each head in the implementation above projects from `d_k` back to `d_model` independently before concatenation, what is the total number of parameters in `MultiHeadCausalAttention` for `d_model=8` and `num_heads=2`? How does this compare to a single `CausalSelfAttention` with `d_k=8`?

### 4.2 PyTorch's built-in

PyTorch provides `nn.MultiheadAttention`, which implements the multi-head mechanism above. The interface differs in a few conventions — the causal mask must be passed explicitly, and its convention inverts ours: `True` marks positions to be *ignored* rather than kept — but the underlying computation is identical to what we have built.

In [ ]:
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, bias=False, batch_first=True)

# generate causal mask: True in positions that should be ignored
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

out_builtin, _ = mha(X, X, X, attn_mask=causal_mask)
print(f"Output shape: {out_builtin.shape}")  # (B, T, d_model)

---
> **Note:** The mechanism we have implemented is conventionally called *causal* attention, and you will encounter that term everywhere. It is worth being precise about why the name is misleading. Causality, in the technical sense, refers to a relationship between events in time: cause precedes effect. Attention masking enforces no such thing. What it enforces is *autoregressivity* — the constraint that the prediction at position $i$ depends only on positions $0$ through $i-1$. This is a constraint on the *computational graph*, not a claim about the causal structure of the data. A model trained autoregressively on text has no privileged access to causal relationships in the world; it simply cannot, by construction, allow future tokens to influence the representation of past ones during training. The mask is a data leakage prevention device. Calling it causal is a category error that has calcified into convention — worth knowing, worth resisting.

> **Think about it.** Q, K, and V were all derived from the same sequence X — that is the only reason this is called *self-attention*. In an encoder-decoder architecture, the queries come from the decoder sequence while the keys and values come from the encoder output; the mechanism is otherwise identical, but it is no longer self-attention because the sequence is querying a different sequence. The encoder's own self-attention layers are also structurally identical to what we built here, with one difference: there is no causal mask. An encoder processes the full sequence in both directions, so every position is free to attend to every other. The mask we applied is strictly a property of autoregressive decoding, not of attention itself.